# Bird Monitoring Debug Notebook

This notebook demonstrates quick function-level debugging for `plugins.bird_monitoring.main`.

Covered examples:
- `normalize_input_files`
- `infer_once` (single image)
- `train_once` (dry-run)


In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "plugins").exists() and (candidate / "platform_core").exists():
            return candidate
    raise RuntimeError("Cannot locate project root")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

from plugins.bird_monitoring.main import normalize_input_files, infer_once, train_once


PROJECT_ROOT: /Users/ronan/Desktop/DarkBreaker


In [ ]:
normalize_input_files("sample.jpg"), normalize_input_files(["a.jpg", "b.jpg"])


In [ ]:
import json
import tempfile
import numpy as np
from PIL import Image

tmp_dir = Path(tempfile.mkdtemp(prefix="bird_nb_"))
img_path = tmp_dir / "single_test.jpg"
out_path = tmp_dir / "infer_summary.json"

img = np.zeros((120, 160, 3), dtype=np.uint8)
img[:, :, 1] = 160
Image.fromarray(img).save(img_path)

infer_summary = infer_once(
    image_paths=[img_path],
    output_json=out_path,
    plugin_config={},
)

print("infer output:", out_path)
print("total_images:", infer_summary["total_images"])
print("first result label:", infer_summary["items"][0]["results"][0]["label"])

print(json.dumps(infer_summary, ensure_ascii=False, indent=2)[:600])


In [ ]:
import yaml

dataset_dir = tmp_dir / "mini_dataset"
(dataset_dir / "images" / "train").mkdir(parents=True, exist_ok=True)
(dataset_dir / "images" / "val").mkdir(parents=True, exist_ok=True)
(dataset_dir / "labels" / "train").mkdir(parents=True, exist_ok=True)
(dataset_dir / "labels" / "val").mkdir(parents=True, exist_ok=True)

train_img = dataset_dir / "images" / "train" / "train_001.jpg"
val_img = dataset_dir / "images" / "val" / "val_001.jpg"
Image.fromarray(img).save(train_img)
Image.fromarray(img).save(val_img)

(dataset_dir / "labels" / "train" / "train_001.txt").write_text("0 0.5 0.5 0.4 0.4\n", encoding="utf-8")
(dataset_dir / "labels" / "val" / "val_001.txt").write_text("0 0.5 0.5 0.4 0.4\n", encoding="utf-8")

data_yaml = dataset_dir / "bird_data.yaml"
with open(data_yaml, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        {
            "train": "images/train",
            "val": "images/val",
            "nc": 1,
            "names": ["bird"],
        },
        f,
        allow_unicode=True,
        sort_keys=False,
    )

train_summary = train_once(
    data_yaml=data_yaml,
    output_dir=tmp_dir / "runs",
    epochs=1,
    imgsz=64,
    batch=1,
    device="cpu",
    run_name="nb_dry_run",
    dry_run=True,
)

print("train backend:", train_summary["backend"])
print("validation valid:", train_summary["validation"]["valid"])
print("summary file:", train_summary["summary_path"])
